# Resume Screening and Job-Fit Classifier

**Course:** Machine Learning (Practical), AI-3202  |  **Lab 14:** Complex Computing Activity  
**Program:** BS Artificial Intelligence, Dawood University of Engineering & Technology  
**Instructor:** Engr. Hamza Farooqui  
**Group:** Muhammad Qasim Khan (24F-AI-015), Muslim Zain (24F-AI-031), Muhammad Ismail (24F-AI-012), Talha Mazhar (24F-AI-022)

---

### Project summary
An end-to-end machine learning system that reads a resume and (1) predicts the job category it fits, (2) groups resumes into skill clusters, and (3) scores how well it matches a job description.

| Item | Details |
|---|---|
| **Dataset** | UpdatedResumeDataSet.csv: 962 rows (166 unique resumes after removing duplicates), 25 job categories |
| **Source** | Originally published on Kaggle; downloaded from a Hugging Face mirror |
| **Models** | Logistic Regression and Naive Bayes (cross-validation tuning) |
| **Clustering** | K-Means |
| **Features** | TF-IDF |
| **Lab concepts** | Labs 1, 3, 9, 10 |
| **Demo** | Streamlit web app (`app.py`) |

**To run:** keep `UpdatedResumeDataSet.csv` in the `Data` folder, then choose **Run All**.

In [ ]:
import pandas as pd, os
df = pd.read_csv("Data/UpdatedResumeDataSet.csv")
os.makedirs("sample_resumes", exist_ok=True)
for cat in ["Data Science", "HR", "Java Developer", "Testing", "Web Designing"]:
    for i, text in enumerate(df[df.Category == cat].Resume.head(2)):
        name = f"sample_resumes/{cat.replace(' ', '_')}_{i+1}.txt"
        open(name, "w", encoding="utf-8").write(text)
print(os.listdir("sample_resumes"))

## 0. Imports and settings

In [ ]:
import os, re, json, joblib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, confusion_matrix,
                             classification_report, silhouette_score, adjusted_rand_score,
                             normalized_mutual_info_score)
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings("ignore")
SEED = 42
BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"
os.makedirs("results", exist_ok=True); os.makedirs("models", exist_ok=True); os.makedirs("data", exist_ok=True)
metrics = {}

def style(ax):
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    ax.grid(axis="y", color="#e5e4e0"); ax.set_axisbelow(True)

## 1. Load data and clean text (Lab 1: preprocessing)

In [ ]:
URL = re.compile(r"https?://\S+|www\.\S+"); EMAIL = re.compile(r"\S+@\S+")
PHONE = re.compile(r"\+?\d[\d\-\s()]{7,}\d"); NON_ALPHA = re.compile(r"[^a-z+#\s]")

def clean_text(text):
    """Lowercase, remove URLs / emails / phone numbers (personal identifiers), punctuation and stop words."""
    t = str(text).lower()
    for pat in (URL, EMAIL, PHONE): t = pat.sub(" ", t)
    t = NON_ALPHA.sub(" ", t)
    return " ".join(w for w in t.split() if w not in ENGLISH_STOP_WORDS and len(w) > 1)

def make_synthetic():
    """Only a fallback so the notebook runs without the real dataset. NEVER report these results."""
    import random; random.seed(1)
    core = {"Data Science": "python pandas numpy machine learning statistics regression tensorflow sql visualization".split(),
            "Web Designing": "html css javascript bootstrap photoshop responsive jquery wordpress ui".split(),
            "Java Developer": "java spring hibernate maven jsp servlet sql microservices junit rest".split(),
            "HR": "recruitment onboarding payroll employee relations training interviewing policy talent".split(),
            "Testing": "selenium manual testing test cases automation junit bug jira qa sql".split()}
    generic = "team communication project responsible experience developed worked managed degree".split()
    rows = []
    for c, ws in core.items():
        for _ in range(60):
            other = random.sample(core[random.choice([k for k in core if k != c])], 2)
            w = random.sample(ws, 5) + other + random.choices(generic, k=10); random.shuffle(w)
            rows.append((" ".join(w), c))
    return pd.DataFrame(rows, columns=["Resume", "Category"])

PATH = "data/UpdatedResumeDataSet.csv"
if os.path.exists(PATH):
    df = pd.read_csv(PATH); print("Loaded REAL dataset:", PATH)
else:
    df = make_synthetic()
    print("!" * 70, "\nWARNING: real dataset not found at", PATH, "\nUsing SYNTHETIC data for testing only. Do NOT use these numbers in your report.\n" + "!" * 70)

n0 = len(df)
df = df.dropna(subset=["Resume", "Category"]).drop_duplicates(subset=["Resume"])
df = df[df["Resume"].str.strip().str.len() > 0].reset_index(drop=True)
metrics["removed_duplicates_or_empty"] = n0 - len(df)
df["clean"] = df["Resume"].apply(clean_text)
df = df[df["clean"].str.len() > 0].reset_index(drop=True)

le = LabelEncoder(); y = le.fit_transform(df["Category"]); classes = list(le.classes_)
metrics.update(n_resumes=len(df), n_categories=len(classes))
print(f"{len(df)} resumes, {len(classes)} categories, removed {metrics['removed_duplicates_or_empty']} duplicate/empty")
df[["Resume", "clean", "Category"]].head()

In [ ]:
counts = df["Category"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(7, 0.32 * len(counts) + 1))
ax.barh(counts.index, counts.values, color=BLUE, height=0.6)
ax.set_xlabel("Number of resumes"); ax.set_title("Class distribution", loc="left"); style(ax)
ax.grid(axis="x", color="#e5e4e0"); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.savefig("results/class_distribution.png", dpi=200); plt.show()
metrics["imbalance_ratio_max_min"] = round(counts.max() / counts.min(), 2)
print("Largest/smallest class ratio:", metrics["imbalance_ratio_max_min"])

## 2. Train/test split (80/20, stratified)

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(df["clean"], y, test_size=0.2, stratify=y, random_state=SEED)
print("Train:", len(X_tr), " Test:", len(X_te))

## 3. Classification models with cross-validation tuning
TF-IDF lives **inside** the pipeline, so each CV fold fits its own vocabulary (no data leakage). The test set is used once, after tuning.

In [ ]:
def make(clf):
    return Pipeline([("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)), ("clf", clf)])

models = {
    "Logistic Regression": (make(LogisticRegression(max_iter=1000)), {"clf__C": [0.1, 1, 10]}),
    "Naive Bayes":         (make(MultinomialNB()),                    {"clf__alpha": [0.1, 0.5, 1.0]}),
}
cv_folds = max(2, min(5, int(np.bincount(y_tr).min())))   # every fold needs one resume of each category
print("Cross-validation folds:", cv_folds)
metrics["cv_folds"] = cv_folds
res, best_est, preds = {}, {}, {}
for name, (pipe, grid) in models.items():
    acc_before = accuracy_score(y_te, pipe.fit(X_tr, y_tr).predict(X_te))      # default settings
    gs = GridSearchCV(pipe, grid, cv=cv_folds, scoring="accuracy", n_jobs=-1).fit(X_tr, y_tr)
    est = gs.best_estimator_; p = est.predict(X_te)
    pr, rc, f1, _ = precision_recall_fscore_support(y_te, p, average="macro", zero_division=0)
    res[name] = dict(best_params={k.split("__")[1]: v for k, v in gs.best_params_.items()},
                     cv_accuracy=round(float(gs.best_score_), 4), acc_before_tuning=round(acc_before, 4),
                     acc_after_tuning=round(accuracy_score(y_te, p), 4),
                     train_accuracy=round(accuracy_score(y_tr, est.predict(X_tr)), 4),
                     precision=round(pr, 4), recall=round(rc, 4), f1=round(f1, 4))
    best_est[name], preds[name] = est, p
results_df = pd.DataFrame(res).T
metrics["classification"] = res
best_name = max(res, key=lambda m: res[m]["cv_accuracy"])     # chosen on CV (training data), not on test
metrics["best_model"] = best_name; best = best_est[best_name]
print(f"Best model (by {cv_folds}-fold CV):", best_name)
results_df

In [ ]:
names = list(res); x = np.arange(len(names)); w = 0.36
fig, ax = plt.subplots(figsize=(7, 4))
b1 = ax.bar(x - w/2, [res[n]["acc_before_tuning"] for n in names], w - 0.04, color=BLUE, label="Before tuning")
b2 = ax.bar(x + w/2, [res[n]["acc_after_tuning"] for n in names], w - 0.04, color=ORANGE, label="After tuning")
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01, f"{b.get_height():.3f}", ha="center", fontsize=8)
ax.set_ylim(0, 1.1); ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylabel("Test accuracy")
ax.set_title("Effect of hyperparameter tuning (cross-validation)", loc="left"); ax.legend(frameon=False); style(ax)
plt.tight_layout(); plt.savefig("results/tuning_effect.png", dpi=200); plt.show()

## 4. Evaluation of the best model: report, confusion matrix, overfitting check

In [ ]:
rep = pd.DataFrame(classification_report(y_te, preds[best_name], target_names=classes, output_dict=True, zero_division=0)).T.round(3)
rep.to_csv("results/per_category_report.csv")
print("Train accuracy:", res[best_name]["train_accuracy"], "| Test accuracy:", res[best_name]["acc_after_tuning"],
      "-> a large gap means overfitting")
rep

In [ ]:
cm = confusion_matrix(y_te, preds[best_name])
fig, ax = plt.subplots(figsize=(0.55 * len(classes) + 3, 0.5 * len(classes) + 2.5))
ax.imshow(cm, cmap="Blues"); ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes, rotation=60, ha="right", fontsize=7); ax.set_yticklabels(classes, fontsize=7)
for i in range(len(classes)):
    for j in range(len(classes)):
        if cm[i, j]: ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7, color="white" if cm[i, j] > cm.max()/2 else INK)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(f"Confusion matrix: {best_name}", loc="left")
plt.tight_layout(); plt.savefig("results/confusion_matrix.png", dpi=200); plt.show()

# most-confused pairs (useful for the interpretation paragraph)
pairs = [(classes[i], classes[j], int(cm[i, j])) for i in range(len(classes)) for j in range(len(classes)) if i != j and cm[i, j] > 0]
print("Most confused (actual -> predicted, count):", sorted(pairs, key=lambda t: -t[2])[:5])

In [ ]:
# Explainability: strongest words per category from Logistic Regression
lr = best_est["Logistic Regression"]; vocab = np.array(lr["tfidf"].get_feature_names_out()); coef = lr["clf"].coef_
top_feats = {c: vocab[np.argsort(coef[i])[-8:][::-1]].tolist() for i, c in enumerate(classes)}
metrics["top_words_per_category"] = top_feats
pd.DataFrame({"Category": classes, "Top words": [", ".join(top_feats[c]) for c in classes]})

## 5. Clustering (Lab 10 K-Means)

In [ ]:
tf = TfidfVectorizer(max_features=3000, stop_words="english", sublinear_tf=True)
Xall = tf.fit_transform(df["clean"]); terms = np.array(tf.get_feature_names_out())
Ks = list(range(2, 31))   # 25 real categories, so search beyond K=12
inertia, sil = [], []
for k in Ks:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Xall)
    inertia.append(km_k.inertia_); sil.append(silhouette_score(Xall, km_k.labels_))
bestK = Ks[int(np.argmax(sil))]
fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))
axs[0].plot(Ks, inertia, color=BLUE, lw=2, marker="o"); axs[0].set_title("Elbow method (inertia)", loc="left")
axs[1].plot(Ks, sil, color=ORANGE, lw=2, marker="o"); axs[1].axvline(bestK, color=MUTED, ls="--", lw=1)
axs[1].set_title(f"Silhouette score (best K = {bestK})", loc="left")
for a in axs: a.set_xlabel("K"); style(a)
plt.tight_layout(); plt.savefig("results/elbow_silhouette.png", dpi=200); plt.show()
print("Best K by silhouette:", bestK, "| silhouette =", round(max(sil), 4))

In [ ]:
km = KMeans(n_clusters=bestK, n_init=10, random_state=SEED).fit(Xall)
cluster_terms = {c: terms[np.argsort(km.cluster_centers_[c])[-8:][::-1]].tolist() for c in range(bestK)}
metrics["clustering"] = dict(best_k=bestK, silhouette=round(max(sil), 4),
    kmeans_ARI=round(adjusted_rand_score(y, km.labels_), 4), kmeans_NMI=round(normalized_mutual_info_score(y, km.labels_), 4), cluster_top_terms=cluster_terms)
print({k: v for k, v in metrics["clustering"].items() if k != "cluster_top_terms"})
for c, t in cluster_terms.items(): print(f"Cluster {c}: {', '.join(t)}")

ct = pd.crosstab(df["Category"], km.labels_)
fig, ax = plt.subplots(figsize=(0.6 * bestK + 4, 0.35 * len(classes) + 2))
ax.imshow(ct.values, cmap="Blues", aspect="auto"); ax.set_xticks(range(bestK)); ax.set_xticklabels([f"C{c}" for c in range(bestK)])
ax.set_yticks(range(len(ct))); ax.set_yticklabels(ct.index, fontsize=7)
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        if ct.values[i, j]: ax.text(j, i, ct.values[i, j], ha="center", va="center", fontsize=6, color="white" if ct.values[i, j] > ct.values.max()/2 else INK)
ax.set_title("Job category vs K-Means cluster", loc="left"); plt.tight_layout()
plt.savefig("results/clusters_vs_categories.png", dpi=200); plt.show()

## 6. Job-fit score (cosine similarity) and keyword-stuffing test

In [ ]:
def fit_score(resume, jd):
    v = tf.transform([clean_text(resume), clean_text(jd)])
    return float(cosine_similarity(v[0], v[1])[0, 0]) * 100

true_cat = df.iloc[0]["Category"]; sample = df.iloc[0]["Resume"]
jd_match = " ".join(top_feats[true_cat]); jd_other = " ".join(top_feats[[c for c in classes if c != true_cat][0]])
demo = dict(true_category=true_cat, strong_match=round(fit_score(sample, jd_match), 1),
            weak_match=round(fit_score(sample, jd_other), 1),
            keyword_stuffed_resume=round(fit_score("team " + jd_match * 3, jd_match), 1))
metrics["fit_score_demo"] = demo; print(demo)
print("A resume that only repeats the job keywords scores very high -> cosine similarity can be gamed (a limitation).")

## 7. Test on resumes you collected yourself (optional but recommended)
Save them as `data/own_resumes.csv` with columns `Resume` and `Category` (category names must match the dataset).

In [ ]:
own = "data/own_resumes.csv"
if os.path.exists(own):
    o = pd.read_csv(own).dropna(); o = o[o["Category"].isin(classes)]
    po = best.predict(o["Resume"].apply(clean_text))
    metrics["own_data_accuracy"] = round(accuracy_score(le.transform(o["Category"]), po), 4); metrics["own_data_n"] = len(o)
    print(f"Accuracy on {len(o)} own resumes:", metrics["own_data_accuracy"])
else:
    print("No data/own_resumes.csv found yet - add your collected resumes and re-run this cell.")

## 8. Demo: paste a resume and a job description

In [ ]:
def analyze(resume, jd):
    r = clean_text(resume); v = tf.transform([r, clean_text(jd)])
    if hasattr(best["clf"], "predict_proba"):
        p = best.predict_proba([r])[0]; idx, conf = int(p.argmax()), f"{p.max()*100:.1f}%"
    else:
        d = best.decision_function([r])[0]; idx, conf = int(d.argmax()), f"margin {d.max():.2f}"
    cl = int(km.predict(v[0])[0]); score = fit_score(resume, jd)
    shared = terms[(v[0].toarray()[0] > 0) & (v[1].toarray()[0] > 0)]
    print(f"Predicted category : {classes[idx]} (confidence {conf})")
    print(f"Skill cluster      : C{cl} -> {', '.join(cluster_terms[cl][:6])}")
    print(f"Job-fit score      : {score:.1f}%")
    print(f"Matching keywords  : {', '.join(shared[:12]) or 'none'}")
    print("Decision support only: a human recruiter makes the final call.")

# Strong match example: a real resume vs a job description built from its own category's key skills
print("--- STRONG MATCH ---"); analyze(df.iloc[0]["Resume"], "We need a candidate with " + jd_match)
# Weak match example: same resume vs a different field
print("\n--- WEAK MATCH ---");  analyze(df.iloc[0]["Resume"], "We need a candidate with " + jd_other)

In [ ]:
# Try your own: paste any resume text and job description between the quotes
my_resume = """Python developer with experience in pandas, scikit-learn, machine learning, SQL and data visualization."""
my_job    = """Looking for a data scientist skilled in python, machine learning, statistics and sql."""
analyze(my_resume, my_job)

## 9. Save model and metrics

In [ ]:
# Both models (Logistic Regression, Naive Bayes) give real class probabilities, so no calibration is needed.
final, final_name = best, best_name
metrics["deployed_model"] = final_name
metrics["deployed_test_accuracy"] = round(accuracy_score(y_te, final.predict(X_te)), 4)
joblib.dump(dict(model=final, label_encoder=le, tfidf=tf, kmeans=km, cluster_terms=cluster_terms, model_name=final_name),
            "models/resume_model.joblib")
json.dump(metrics, open("results/metrics.json", "w"), indent=2, default=str)
print("Saved models/resume_model.joblib and results/metrics.json. Deployed model:", final_name)